# M1 최소학습 baseline

## tl;dr

M1 7일 이벤트 feature 50행으로 `normal`과 `efd_possible` 최소 baseline을 검증했다. Dummy는 다수 클래스인 `normal`만 예측해 accuracy는 0.6976이지만 `efd_possible` recall은 0이다. Logistic baseline은 accuracy는 0.5927로 낮지만 balanced accuracy 0.5946, recall 0.6000, f1 0.4714로 positive를 일부 잡았다. 현재 feature에 약한 분류 신호가 있을 가능성은 있으나, 50행 실험이므로 성능 일반화 주장은 금물이다.


## Context & Methods

### Key Assumptions

- 입력은 `07_데이터산출물/m1_classification_features.csv`만 사용한다.
- 학습 feature는 센서 통계 컬럼 70개만 사용한다. 메타데이터, 날짜, 이벤트 ID, `substation_id`, `coverage_rate`는 모델 입력에서 제외한다.
- 같은 기계실의 이벤트가 학습/평가에 동시에 들어가면 누수 위험이 있으므로 `substation_id`를 group으로 사용한다.
- 데이터가 50행뿐이므로 성능 수치는 참고용이다. 주 판단 기준은 Dummy baseline 대비 개선 여부와 오분류 패턴이다.


In [1]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
INPUT_PATH = ROOT / '07_데이터산출물' / 'm1_classification_features.csv'
METRICS_PATH = ROOT / '07_데이터산출물' / 'm1_baseline_cv_metrics.csv'
PREDICTIONS_PATH = ROOT / '07_데이터산출물' / 'm1_baseline_cv_predictions.csv'
REPORT_PATH = ROOT / '07_데이터산출물' / '03_M1_최소학습_baseline_보고서.md'
RANDOM_STATE = 42
POSITIVE_LABEL = 'efd_possible'


## Data

### 1. Load and Validate Input


In [2]:
features = pd.read_csv(INPUT_PATH)
feature_columns = [column for column in features.columns if '__' in column]
metadata_columns = [column for column in features.columns if '__' not in column]

assert len(features) == 50, len(features)
assert features['label'].value_counts().to_dict() == {'normal': 35, 'efd_possible': 15}
assert set(features['manufacturer']) == {'manufacturer_1'}
assert len(feature_columns) == 70, len(feature_columns)
assert not {'substation_id', 'source_event_id', 'coverage_rate'}.intersection(feature_columns)

data_summary = pd.DataFrame([
    {'item': 'rows', 'value': len(features)},
    {'item': 'feature_columns', 'value': len(feature_columns)},
    {'item': 'normal', 'value': int((features['label'] == 'normal').sum())},
    {'item': 'efd_possible', 'value': int((features['label'] == 'efd_possible').sum())},
    {'item': 'substations', 'value': int(features['substation_id'].nunique())},
])
data_summary


,item,value
0,rows,50
1,feature_columns,70
2,normal,35
3,efd_possible,15
4,substations,29


In [3]:
quality_summary = features.groupby('label').agg(
    rows=('sample_id', 'count'),
    substations=('substation_id', 'nunique'),
    coverage_min=('coverage_rate', 'min'),
    coverage_median=('coverage_rate', 'median'),
    coverage_max=('coverage_rate', 'max'),
).round(4)
quality_summary


,rows,substations,coverage_min,coverage_median,coverage_max
label,,,,,
efd_possible,15,15,0.7242,1.0,1.0
normal,35,22,1.0000,1.0,1.0


## Results

### 2. Configure Group CV and Models


In [4]:
X = features[feature_columns]
y = (features['label'] == POSITIVE_LABEL).astype(int)
groups = features['substation_id'].astype(int)

# 5 folds keeps the test groups reasonably small while each fold still has a chance to contain both labels.
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
splits = list(cv.split(X, y, groups))

models = {
    'dummy_most_frequent': DummyClassifier(strategy='most_frequent'),
    'logistic_balanced': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
    ]),
}

fold_overview = []
for fold, (_, test_index) in enumerate(splits, start=1):
    y_test = y.iloc[test_index]
    fold_overview.append({
        'fold': fold,
        'test_rows': int(len(test_index)),
        'test_groups': int(groups.iloc[test_index].nunique()),
        'test_normal': int((y_test == 0).sum()),
        'test_efd_possible': int((y_test == 1).sum()),
    })
fold_overview = pd.DataFrame(fold_overview)
fold_overview


,fold,test_rows,test_groups,test_normal,test_efd_possible
0,1,9,5,6,3
1,2,9,5,6,3
2,3,11,6,8,3
3,4,11,7,8,3
4,5,10,6,7,3


### 3. Run Cross-Validation


In [5]:
def score_fold(model_name: str, y_true: pd.Series, y_pred: np.ndarray, y_score: np.ndarray | None) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result = {
        'model': model_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': np.nan,
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }
    if y_score is not None and y_true.nunique() == 2:
        result['roc_auc'] = roc_auc_score(y_true, y_score)
    return result

metric_rows = []
prediction_rows = []

for fold, (train_index, test_index) in enumerate(splits, start=1):
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]
    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        if hasattr(model, 'predict_proba'):
            y_score = model.predict_proba(X_test)[:, 1]
        else:
            y_score = None

        scores = score_fold(model_name, y_test, y_pred, y_score)
        scores.update({
            'fold': fold,
            'train_rows': int(len(train_index)),
            'test_rows': int(len(test_index)),
            'train_groups': int(groups.iloc[train_index].nunique()),
            'test_groups': int(groups.iloc[test_index].nunique()),
            'test_normal': int((y_test == 0).sum()),
            'test_efd_possible': int((y_test == 1).sum()),
        })
        metric_rows.append(scores)

        fold_predictions = features.iloc[test_index][[
            'sample_id', 'label', 'substation_id', 'source_event_id', 'window_start', 'window_end', 'coverage_rate'
        ]].copy()
        fold_predictions['fold'] = fold
        fold_predictions['model'] = model_name
        fold_predictions['actual_binary'] = y_test.to_numpy()
        fold_predictions['predicted_binary'] = y_pred
        fold_predictions['predicted_label'] = np.where(y_pred == 1, 'efd_possible', 'normal')
        fold_predictions['positive_score'] = y_score if y_score is not None else np.nan
        fold_predictions['is_correct'] = fold_predictions['actual_binary'] == fold_predictions['predicted_binary']
        prediction_rows.append(fold_predictions)

metrics = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_rows, ignore_index=True)
metrics.to_csv(METRICS_PATH, index=False, encoding='utf-8-sig')
predictions.to_csv(PREDICTIONS_PATH, index=False, encoding='utf-8-sig')
metrics


,model,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,tn,fp,fn,tp,fold,train_rows,test_rows,train_groups,test_groups,test_normal,test_efd_possible
0,dummy_most_frequent,0.666667,0.500000,0.00,0.000000,0.000000,0.500000,6,0,3,0,1,41,9,24,5,6,3
1,logistic_balanced,0.555556,0.583333,0.40,0.666667,0.500000,0.666667,3,3,1,2,1,41,9,24,5,6,3
2,dummy_most_frequent,0.666667,0.500000,0.00,0.000000,0.000000,0.500000,6,0,3,0,2,41,9,24,5,6,3
3,logistic_balanced,0.444444,0.416667,0.25,0.333333,0.285714,0.500000,3,3,2,1,2,41,9,24,5,6,3
4,dummy_most_frequent,0.727273,0.500000,0.00,0.000000,0.000000,0.500000,8,0,3,0,3,39,11,23,6,8,3
5,logistic_balanced,0.636364,0.645833,0.40,0.666667,0.500000,0.583333,5,3,1,2,3,39,11,23,6,8,3
6,dummy_most_frequent,0.727273,0.500000,0.00,0.000000,0.000000,0.500000,8,0,3,0,4,39,11,22,7,8,3
7,logistic_balanced,0.727273,0.708333,0.50,0.666667,0.571429,0.666667,6,2,1,2,4,39,11,22,7,8,3
8,dummy_most_frequent,0.700000,0.500000,0.00,0.000000,0.000000,0.500000,7,0,3,0,5,40,10,23,6,7,3
9,logistic_balanced,0.600000,0.619048,0.40,0.666667,0.500000,0.619048,4,3,1,2,5,40,10,23,6,7,3


In [6]:
summary_metrics = metrics.groupby('model').agg(
    accuracy_mean=('accuracy', 'mean'),
    accuracy_std=('accuracy', 'std'),
    balanced_accuracy_mean=('balanced_accuracy', 'mean'),
    balanced_accuracy_std=('balanced_accuracy', 'std'),
    precision_mean=('precision', 'mean'),
    recall_mean=('recall', 'mean'),
    f1_mean=('f1', 'mean'),
    roc_auc_mean=('roc_auc', 'mean'),
).round(4)
summary_metrics


,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,precision_mean,recall_mean,f1_mean,roc_auc_mean
model,,,,,,,,
dummy_most_frequent,0.6976,0.0303,0.5000,0.0000,0.00,0.0,0.0000,0.5000
logistic_balanced,0.5927,0.1042,0.5946,0.1095,0.39,0.6,0.4714,0.6071


### 4. Inspect Logistic Coefficients


In [7]:
final_logistic = models['logistic_balanced']
final_logistic.fit(X, y)
coefs = final_logistic.named_steps['model'].coef_[0]
coef_table = pd.DataFrame({'feature': feature_columns, 'coefficient': coefs})
positive_features = coef_table.sort_values('coefficient', ascending=False).head(10).reset_index(drop=True)
negative_features = coef_table.sort_values('coefficient', ascending=True).head(10).reset_index(drop=True)
positive_features


,feature,coefficient
0,outdoor_temperature__last_minus_first,0.654015
1,p_net_supply_temperature__max,0.653581
2,p_hc1_return_temperature__mean,0.611841
3,p_hc1_return_temperature__median,0.558334
4,p_net_meter_heat_power__max,0.500940
5,outdoor_temperature__max,0.482182
6,p_hc1_return_temperature__std,0.396115
7,s_hc1_supply_temperature_setpoint__min,0.363397
8,outdoor_temperature__std,0.337016
9,p_net_meter_heat_power__mean,0.317617


In [8]:
negative_features


,feature,coefficient
0,s_hc1_supply_temperature__min,-0.602084
1,p_net_meter_flow__max,-0.561855
2,p_net_supply_temperature__min,-0.516821
3,p_hc1_return_temperature__min,-0.516662
4,s_hc1_supply_temperature__last_minus_first,-0.495904
5,p_net_meter_flow__min,-0.489624
6,p_net_return_temperature__min,-0.456421
7,p_net_return_temperature__last_minus_first,-0.447715
8,p_hc1_return_temperature__last_minus_first,-0.423797
9,s_hc1_supply_temperature_setpoint__median,-0.380818


### 5. Review Misclassifications


In [9]:
misclassified = predictions[(predictions['model'] == 'logistic_balanced') & (~predictions['is_correct'])].copy()
misclassified.sort_values(['fold', 'sample_id'])


,sample_id,label,substation_id,source_event_id,window_start,window_end,coverage_rate,fold,model,actual_binary,predicted_binary,predicted_label,positive_score,is_correct
9,efd_possible_0003,efd_possible,12,3,2015-11-24 10:56:00,2015-12-01 10:56:00,1.000000,1,logistic_balanced,1,0,normal,0.001956,False
12,normal_0027,normal,12,27,2015-06-24 13:34:00,2015-07-01 13:34:00,1.000000,1,logistic_balanced,0,1,efd_possible,0.554686,False
16,normal_0042,normal,22,42,2019-11-01 00:00:00,2019-11-08 00:00:00,1.000000,1,logistic_balanced,0,1,efd_possible,0.786022,False
17,normal_0054,normal,22,54,2020-03-14 00:00:00,2020-03-21 00:00:00,1.000000,1,logistic_balanced,0,1,efd_possible,0.904147,False
29,efd_possible_0049,efd_possible,18,49,2019-04-27 07:19:00,2019-05-04 07:19:00,0.997024,2,logistic_balanced,1,0,normal,0.212645,False
27,efd_possible_0052,efd_possible,21,52,2016-12-05 15:55:00,2016-12-12 15:55:00,1.000000,2,logistic_balanced,1,0,normal,0.000863,False
35,normal_0008,normal,6,8,2020-06-02 00:00:00,2020-06-09 00:00:00,1.000000,2,logistic_balanced,0,1,efd_possible,0.795799,False
32,normal_0017,normal,2,17,2018-02-27 00:00:00,2018-03-06 00:00:00,1.000000,2,logistic_balanced,0,1,efd_possible,0.633468,False
33,normal_0056,normal,2,56,2018-11-29 00:00:00,2018-12-06 00:00:00,1.000000,2,logistic_balanced,0,1,efd_possible,0.600765,False
47,efd_possible_0044,efd_possible,8,44,2018-04-20 08:15:00,2018-04-27 08:15:00,1.000000,3,logistic_balanced,1,0,normal,0.231208,False


## Takeaways

### 6. Write Report and Validate Outputs


In [10]:
def fmt_float(value):
    if pd.isna(value):
        return ''
    return f'{value:.4f}'

def markdown_table(df: pd.DataFrame) -> str:
    if df.empty:
        return ''
    clean = df.copy().where(pd.notna(df), '')
    headers = [str(column) for column in clean.columns]
    rows = [[str(value) for value in row] for row in clean.to_numpy()]

    def escape(value: str) -> str:
        return value.replace('|', '\|').replace('\n', '<br>')

    lines = [
        '| ' + ' | '.join(escape(header) for header in headers) + ' |',
        '| ' + ' | '.join('---' for _ in headers) + ' |',
    ]
    for row in rows:
        lines.append('| ' + ' | '.join(escape(value) for value in row) + ' |')
    return '\n'.join(lines)

summary_for_report = summary_metrics.reset_index()
fold_for_report = fold_overview.copy()
misclassified_for_report = misclassified[[
    'fold', 'sample_id', 'label', 'predicted_label', 'substation_id', 'source_event_id', 'coverage_rate', 'positive_score'
]].copy()
for column in ['coverage_rate', 'positive_score']:
    if column in misclassified_for_report.columns:
        misclassified_for_report[column] = misclassified_for_report[column].map(fmt_float)

report = f'''# M1 최소학습 baseline 보고서

## 개요

M1 전용 전처리 데이터셋 50행으로 `normal`과 `efd_possible`을 구분하는 최소 baseline을 실행했다. 이번 단계의 목적은 운영 모델 확정이 아니라, 현재 feature에 분류 신호가 있는지와 검증 방식이 안전한지 확인하는 것이다.

## 데이터

| 항목 | 값 |
|---|---:|
| 전체 샘플 | {len(features)} |
| normal | {int((features['label'] == 'normal').sum())} |
| efd_possible | {int((features['label'] == 'efd_possible').sum())} |
| 학습 feature | {len(feature_columns)} |
| 고유 substation | {int(features['substation_id'].nunique())} |

## 학습/평가 방식

- metadata, 날짜, 이벤트 ID, `substation_id`, `coverage_rate`는 모델 입력에서 제외했다.
- 센서 통계 feature 70개만 사용했다.
- 같은 기계실이 train/test에 동시에 들어가지 않도록 `substation_id` 기준 group CV를 사용했다.
- 비교 기준은 `DummyClassifier(strategy="most_frequent")`와 `StandardScaler + LogisticRegression(class_weight="balanced")`다.

## Fold 구성

{markdown_table(fold_for_report)}

## CV 결과

{markdown_table(summary_for_report)}

## Logistic positive 방향 상위 feature

{markdown_table(positive_features)}

## Logistic negative 방향 상위 feature

{markdown_table(negative_features)}

## Logistic 오분류 샘플

{markdown_table(misclassified_for_report) if len(misclassified_for_report) else '오분류 없음'}

## 해석

- 이 결과는 50행짜리 최소 baseline이므로 성능 일반화 근거로 사용하면 안 된다.
- Dummy는 다수 클래스인 `normal`만 예측해 accuracy는 0.6976이지만, `efd_possible` recall과 f1은 0이다.
- Logistic은 accuracy는 0.5927로 낮지만, balanced accuracy 0.5946, recall 0.6000, f1 0.4714로 positive를 일부 잡는다.
- 따라서 현재 feature에는 약한 분류 신호가 있을 수 있지만, 아직 모델 성능을 주장할 단계는 아니다.
- positive가 15건뿐이라 fold별 recall과 f1 변동이 클 수 있다.
- Event ID 20은 coverage가 0.7242라 품질 주의가 필요하지만, positive 샘플 수가 적어 이번 baseline에서는 유지했다.

## 다음 단계

- fold별 오분류를 보고 feature window 또는 label 기준을 조정할지 판단한다.
- 성능 주장을 하기 전에 샘플 수 확장 또는 label 정책 확장 실험을 분리해서 진행한다.
'''
REPORT_PATH.write_text(report, encoding='utf-8')

assert METRICS_PATH.exists()
assert PREDICTIONS_PATH.exists()
assert REPORT_PATH.exists()
assert len(metrics) == 10
assert set(metrics['model']) == {'dummy_most_frequent', 'logistic_balanced'}
assert len(predictions) == 100
assert set(predictions['model']) == {'dummy_most_frequent', 'logistic_balanced'}
assert len(feature_columns) == 70
assert not {'substation_id', 'source_event_id', 'coverage_rate'}.intersection(feature_columns)

print('metrics:', METRICS_PATH.relative_to(ROOT))
print('predictions:', PREDICTIONS_PATH.relative_to(ROOT))
print('report:', REPORT_PATH.relative_to(ROOT))
print('validation complete')


metrics: 07_데이터산출물\m1_baseline_cv_metrics.csv
predictions: 07_데이터산출물\m1_baseline_cv_predictions.csv
report: 07_데이터산출물\03_M1_최소학습_baseline_보고서.md
validation complete


<>:14: SyntaxWarning: invalid escape sequence '\|'
<>:14: SyntaxWarning: invalid escape sequence '\|'
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26796\4213473318.py:14: SyntaxWarning: invalid escape sequence '\|'
  return value.replace('|', '\|').replace('\n', '<br>')
